In [5]:
import json
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt

# --- EDIT THIS ---
COCO_JSON = Path("/zpool/vladlab/data_drive/stimulus_sets/geogaze_COCO_stim/coco_working/working_v3/instances_train_filtered3.json")  


In [6]:
with COCO_JSON.open("r") as f:
    coco = json.load(f)

images = coco.get("images", [])
annotations = coco.get("annotations", [])
categories = coco.get("categories", [])

print("Loaded:")
print("  images:", len(images))
print("  annotations:", len(annotations))
print("  categories:", len(categories))


Loaded:
  images: 33953
  annotations: 60549
  categories: 80


In [12]:
# Category lookup
cat_id_to_name = {c["id"]: c["name"] for c in categories}
cat_name_to_id = {c["name"]: c["id"] for c in categories}

# Image lookup
img_id_to_info = {im["id"]: im for im in images}  # contains width/height/file_name/etc

# Quick sanity check: annotations that reference missing images/categories
missing_img = sum(1 for a in annotations if a["image_id"] not in img_id_to_info)
missing_cat = sum(1 for a in annotations if a["category_id"] not in cat_id_to_name)

print("Sanity checks:")
print("  annotations w/ missing image_id:", missing_img)
print("  annotations w/ missing category_id:", missing_cat)


Sanity checks:
  annotations w/ missing image_id: 0
  annotations w/ missing category_id: 0


In [13]:
# Count annotations per image_id
ann_count_by_image = Counter(a["image_id"] for a in annotations)

# Include images that have 0 annotations
all_image_ids = [im["id"] for im in images]
boxes_per_image = [ann_count_by_image.get(img_id, 0) for img_id in all_image_ids]

dist = Counter(boxes_per_image)  # key: #boxes, value: #images

dist_df = (
    pd.DataFrame(sorted(dist.items()), columns=["n_boxes", "n_images"])
      .assign(pct_images=lambda d: d["n_images"] / d["n_images"].sum() * 100)
)

dist_df.head(6)


,n_boxes,n_images,pct_images
0,1,36712,43.247064
1,2,25449,29.979149
2,3,13180,15.526158
3,4,6593,7.766613
4,5,2955,3.481016


In [14]:
# For each category, keep a set of image_ids where it appears
cat_to_images = defaultdict(set)
for a in annotations:
    cid = a["category_id"]
    imgid = a["image_id"]
    if cid in cat_id_to_name and imgid in img_id_to_info:
        cat_to_images[cid].add(imgid)

images_per_cat = [
    {"category_id": cid, "category_name": cat_id_to_name.get(cid, f"UNK_{cid}"), "n_images": len(img_set)}
    for cid, img_set in cat_to_images.items()
]
images_per_cat_df = pd.DataFrame(images_per_cat).sort_values("n_images", ascending=False)

images_per_cat_df.head(80)


,category_id,category_name,n_images
18,1,person,38160
11,67,dining table,8387
10,62,chair,4215
22,65,bed,3388
4,17,cat,3347
...,...,...,...
50,74,mouse,148
69,40,baseball glove,65
70,37,sports ball,47
55,80,toaster,37


In [15]:
ann_per_cat = Counter(a["category_id"] for a in annotations if a["category_id"] in cat_id_to_name)

ann_per_cat_df = (
    pd.DataFrame(
        [{"category_id": cid, "category_name": cat_id_to_name[cid], "n_annotations": n}
         for cid, n in ann_per_cat.items()]
    )
    .sort_values("n_annotations", ascending=False)
)

ann_per_cat_df.head(20)


,category_id,category_name,n_annotations
18,1,person,56924
11,67,dining table,8713
10,62,chair,5011
9,63,couch,3903
22,65,bed,3677
4,17,cat,3652
33,25,giraffe,3488
13,3,car,3061
0,18,dog,3048
16,6,bus,2939
